# PENGWIN Viewer

Interactive notebook viewer for the shared PENGWIN dataset.

- Task 2 X-ray works with the current environment.
- Task 1 CT requires `SimpleITK` to be installed.
- The viewer lets you switch cases, toggle overlays, and adjust opacity.


In [1]:
from pathlib import Path

%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from PIL import Image

try:
    import SimpleITK as sitk
except ImportError:
    sitk = None

WORKSPACE_ROOT = Path('/gpfs/home5/scur0509/projects')
DATA_ROOT = WORKSPACE_ROOT / 'data' / 'pengwin' / 'original'

XRAY_IMAGE_ROOT = DATA_ROOT / 'task2_xray' / 'train' / 'input' / 'images' / 'x-ray'
XRAY_LABEL_ROOT = DATA_ROOT / 'task2_xray' / 'train' / 'output' / 'images' / 'x-ray'
CT_IMAGE_ROOT = DATA_ROOT / 'task1_ct' / 'images'
CT_LABEL_ROOT = DATA_ROOT / 'task1_ct' / 'labels'

XRAY_CATEGORIES = {
    1: 'SA',
    2: 'LI',
    3: 'RI',
}
XRAY_COLORS = {
    1: np.array([0.93, 0.34, 0.23]),
    2: np.array([0.17, 0.63, 0.17]),
    3: np.array([0.21, 0.49, 0.86]),
}

def list_xray_cases():
    return sorted(path.stem for path in XRAY_IMAGE_ROOT.glob('*.tif'))

def list_ct_cases():
    return sorted(path.stem for path in CT_IMAGE_ROOT.glob('*.mha'))

def decode_xray_segmentation(segmentation):
    overlay = np.zeros(segmentation.shape + (4,), dtype=np.float32)
    legend_items = []
    for category_id, label in XRAY_CATEGORIES.items():
        category_mask = np.zeros(segmentation.shape, dtype=bool)
        for fragment_id in range(1, 11):
            shift = 10 * (category_id - 1) + fragment_id
            mask = ((segmentation >> shift) & 1).astype(bool)
            if np.any(mask):
                category_mask |= mask
                legend_items.append(f'{label}-{fragment_id}')
        if np.any(category_mask):
            overlay[category_mask, :3] = XRAY_COLORS[category_id]
            overlay[category_mask, 3] = 1.0
    return overlay, legend_items

def normalize_xray_image(image):
    image = image.astype(np.float32)
    image = image - image.min()
    denom = image.max()
    if denom > 0:
        image = image / denom
    image = -np.log(image + 1e-2)
    image = image - image.min()
    denom = image.max()
    if denom > 0:
        image = image / denom
    return image

def load_xray_case(case_id):
    image_path = XRAY_IMAGE_ROOT / f'{case_id}.tif'
    label_path = XRAY_LABEL_ROOT / f'{case_id}.tif'
    image = np.array(Image.open(image_path))
    segmentation = np.array(Image.open(label_path))
    overlay, legend_items = decode_xray_segmentation(segmentation)
    return {
        'image': normalize_xray_image(image),
        'overlay': overlay,
        'legend': ', '.join(legend_items) if legend_items else 'No labels',
        'title': f'Task 2 X-ray: {case_id}',
    }

def load_ct_case(case_id):
    if sitk is None:
        raise RuntimeError('CT viewing requires SimpleITK. Install it with `pip install SimpleITK`.')
    image_path = CT_IMAGE_ROOT / f'{case_id}.mha'
    label_path = CT_LABEL_ROOT / f'{case_id}.mha'
    image = sitk.GetArrayFromImage(sitk.ReadImage(str(image_path))).astype(np.float32)
    labels = sitk.GetArrayFromImage(sitk.ReadImage(str(label_path))).astype(np.int32)
    image = np.clip(image, -1000, 1500)
    image = image - image.min()
    denom = image.max()
    if denom > 0:
        image = image / denom
    overlay = np.zeros(labels.shape + (4,), dtype=np.float32)
    mask = labels > 0
    overlay[mask, :3] = np.array([0.95, 0.29, 0.20], dtype=np.float32)
    overlay[mask, 3] = 1.0
    label_values = np.unique(labels)
    label_values = label_values[label_values > 0]
    legend = f'Labels present: {len(label_values)}'
    if len(label_values):
        preview = ', '.join(map(str, label_values[:12]))
        legend += f' | ids: {preview}'
        if len(label_values) > 12:
            legend += ', ...'
    return {
        'image': image,
        'overlay': overlay,
        'legend': legend,
        'title': f'Task 1 CT: {case_id}',
    }


In [2]:
task_dropdown = widgets.Dropdown(
    options=[('X-ray', 'xray'), ('CT', 'ct')],
    value='xray',
    description='Task:',
)

case_dropdown = widgets.Dropdown(description='Case:')
overlay_checkbox = widgets.Checkbox(value=True, description='Show overlay')
alpha_slider = widgets.FloatSlider(value=0.35, min=0.0, max=1.0, step=0.05, description='Alpha:')
slice_slider = widgets.IntSlider(value=0, min=0, max=0, step=1, description='Slice:')
status = widgets.HTML()

out = widgets.Output()

def update_case_options(*_):
    if task_dropdown.value == 'xray':
        cases = list_xray_cases()
        slice_slider.layout.display = 'none'
    else:
        cases = list_ct_cases()
        slice_slider.layout.display = ''
    case_dropdown.options = cases
    if cases:
        case_dropdown.value = cases[0]

task_dropdown.observe(update_case_options, names='value')
update_case_options()

controls = widgets.VBox([
    widgets.HBox([task_dropdown, case_dropdown]),
    widgets.HBox([overlay_checkbox, alpha_slider]),
    slice_slider,
    status,
])
display(controls, out)


Output()

In [3]:
def render(*_):
    with out:
        out.clear_output(wait=True)
        try:
            if task_dropdown.value == 'xray':
                case = load_xray_case(case_dropdown.value)
                image = case['image']
                overlay = case['overlay']
                status.value = f"<b>{case['title']}</b><br>{case['legend']}"
                fig, ax = plt.subplots(figsize=(8, 8))
                ax.imshow(image, cmap='gray')
                if overlay_checkbox.value:
                    ax.imshow(overlay, alpha=alpha_slider.value)
                ax.set_axis_off()
                ax.set_title(case_dropdown.value)
                display(fig)
                plt.close(fig)
            else:
                case = load_ct_case(case_dropdown.value)
                max_slice = case['image'].shape[0] - 1
                slice_slider.max = max_slice
                slice_idx = min(slice_slider.value, max_slice)
                image = case['image'][slice_idx]
                overlay = case['overlay'][slice_idx]
                status.value = f"<b>{case['title']}</b><br>{case['legend']}<br>Slice {slice_idx + 1}/{max_slice + 1}"
                fig, ax = plt.subplots(figsize=(8, 8))
                ax.imshow(image, cmap='gray')
                if overlay_checkbox.value:
                    ax.imshow(overlay, alpha=alpha_slider.value)
                ax.set_axis_off()
                ax.set_title(case_dropdown.value)
                display(fig)
                plt.close(fig)
        except Exception as exc:
            status.value = f"<b>Error:</b> {exc}"

for widget in [task_dropdown, case_dropdown, overlay_checkbox, alpha_slider, slice_slider]:
    widget.observe(render, names='value')

render()
